# Perspective Rectification and Scale

Perspective correction is useful when a camera views a flat surface at an angle. If you know four points on a rectangular reference object, a homography can transform the image into a top-down view. Once the rectangle has known real-world dimensions, pixel measurements can be converted into physical units.

This notebook uses a synthetic example so it can run without private data. Replace the four source points with points clicked or measured from your own frame.

**Requirements:** `opencv-python`, `numpy`, `matplotlib`


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})


## Define a reference rectangle

The source points are the four visible corners in the distorted camera image. The destination points define the corrected top-down rectangle. Keep point order consistent: top-left, top-right, bottom-right, bottom-left.

For real data, measure or click the source points in image coordinates and set the real width/height in centimeters, millimeters, or another unit.


In [ ]:
# Synthetic image size
h, w = 720, 960
img = np.full((h, w, 3), 245, dtype=np.uint8)

# A skewed rectangle as it appears in the camera frame.
src_pts = np.array([
    [260, 160],  # top-left
    [730, 115],  # top-right
    [820, 560],  # bottom-right
    [170, 610],  # bottom-left
], dtype=np.float32)

# Draw a grid inside the reference plane, then warp it into the camera image.
rect_w_px, rect_h_px = 640, 420
plane = np.full((rect_h_px, rect_w_px, 3), 235, dtype=np.uint8)
for x in range(0, rect_w_px + 1, 80):
    cv2.line(plane, (x, 0), (x, rect_h_px), (90, 90, 90), 1)
for y in range(0, rect_h_px + 1, 60):
    cv2.line(plane, (0, y), (rect_w_px, y), (90, 90, 90), 1)
cv2.rectangle(plane, (0, 0), (rect_w_px - 1, rect_h_px - 1), (30, 30, 30), 3)
cv2.circle(plane, (460, 250), 36, (40, 120, 220), -1)

dst_corners = np.array(
    [[0, 0], [rect_w_px - 1, 0], [rect_w_px - 1, rect_h_px - 1], [0, rect_h_px - 1]],
    dtype=np.float32,
)
H_to_camera = cv2.getPerspectiveTransform(dst_corners, src_pts)
warped_plane = cv2.warpPerspective(plane, H_to_camera, (w, h))
mask = cv2.warpPerspective(
    np.full((rect_h_px, rect_w_px), 255, dtype=np.uint8), H_to_camera, (w, h)
)
img[mask > 0] = warped_plane[mask > 0]

plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.scatter(src_pts[:, 0], src_pts[:, 1], c="red", s=35)
for i, (x, y) in enumerate(src_pts, start=1):
    plt.text(x + 8, y - 8, str(i), color="red", weight="bold")
plt.title("Camera view with four reference corners")
plt.axis("off");


## Rectify the image

`cv2.getPerspectiveTransform` estimates the homography from the four source corners to the desired top-down rectangle. `cv2.warpPerspective` applies that transformation to the whole image.


In [ ]:
real_width_cm = 32.0
real_height_cm = 21.0
px_per_cm = 20

out_w = int(real_width_cm * px_per_cm)
out_h = int(real_height_cm * px_per_cm)
dst_pts = np.array(
    [[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]],
    dtype=np.float32,
)

H = cv2.getPerspectiveTransform(src_pts, dst_pts)
rectified = cv2.warpPerspective(img, H, (out_w, out_h))

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axs[0].plot(*np.vstack([src_pts, src_pts[0]]).T, color="red", lw=2)
axs[0].set_title("Original camera view")
axs[0].axis("off")
axs[1].imshow(cv2.cvtColor(rectified, cv2.COLOR_BGR2RGB))
axs[1].set_title("Rectified top-down view")
axs[1].axis("off")
plt.tight_layout()

print(f"Output size: {out_w} x {out_h} px")
print(f"Scale: {px_per_cm:.1f} px/cm, or {1 / px_per_cm:.3f} cm/px")


## Convert pixel measurements to real-world units

Once the image is rectified, distances on the reference plane can be measured in pixels and converted to real-world units using the known scale. This works for objects lying on the same plane as the reference rectangle.


In [ ]:
# Example measurement in rectified coordinates.
point_a = np.array([385, 225], dtype=float)
point_b = np.array([530, 285], dtype=float)

distance_px = np.linalg.norm(point_b - point_a)
distance_cm = distance_px / px_per_cm

plt.figure(figsize=(7, 4.6))
plt.imshow(cv2.cvtColor(rectified, cv2.COLOR_BGR2RGB))
plt.plot([point_a[0], point_b[0]], [point_a[1], point_b[1]], color="yellow", lw=3)
plt.scatter([point_a[0], point_b[0]], [point_a[1], point_b[1]], c="yellow", s=40)
plt.title(f"Measured distance: {distance_cm:.2f} cm")
plt.axis("off");

print(f"Distance: {distance_px:.1f} px = {distance_cm:.2f} cm")


## Adapting this notebook

For real data, replace `img` with a frame loaded by `cv2.imread`, set `src_pts` to the four reference corners, and enter the real-world reference dimensions. Save generated rectified images or videos into an ignored output folder such as `notebook_outputs/` or `undistorted/`.
